In [2]:
# =========================================================
# 🩺 Stroke Risk Prediction - Model Training Pipeline
# =========================================================

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------
# Ensure 'healthcare-dataset-stroke-data.csv' is uploaded to Colab
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print("Dataset Shape:", df.shape)

# ---------------------------------------------------------
# 2. Data Cleaning & Preprocessing
# ---------------------------------------------------------
# Drop irrelevant ID column
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Handle missing values in BMI with Median
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Handle 'Other' gender if present
df = df[df['gender'] != 'Other']

# ---------------------------------------------------------
# 3. Categorical Feature Encoding (One-Hot Encoding)
# ---------------------------------------------------------
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate Features (X) and Target (y)
X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']

# Save Feature Columns List for Streamlit App alignment
model_columns = list(X.columns)
joblib.dump(model_columns, 'model_columns.pkl')
print("Model Columns Saved! Total Features:", len(model_columns))

# ---------------------------------------------------------
# 4. Train - Test Split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 5. Model Training (Balanced Random Forest)
# ---------------------------------------------------------
# Using class_weight='balanced' to handle stroke class imbalance
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. Model Evaluation
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n--- Model Performance ---")
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------------------------------------
# 7. Save Trained Model
# ---------------------------------------------------------
joblib.dump(model, 'stroke_model.pkl', compress=3)
print("✅ Stroke Prediction Model successfully saved as 'stroke_model.pkl'!")

Dataset Shape: (5110, 12)
Model Columns Saved! Total Features: 15

--- Model Performance ---
Accuracy Score: 0.9197651663405088
ROC AUC Score: 0.7991666666666667

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.96      0.96       972
           1       0.18      0.18      0.18        50

    accuracy                           0.92      1022
   macro avg       0.57      0.57      0.57      1022
weighted avg       0.92      0.92      0.92      1022

✅ Stroke Prediction Model successfully saved as 'stroke_model.pkl'!
